# Cerro Machín synthetic experiment — compositional DIP

This notebook is a **self-contained reproduction of our method only** for the Cerro Machín synthetic gravity experiment (`modelA`). All implementation code needed by the experiment is included below: data loading, synthetic-data generation, depth weighting, Fourier-feature networks, compositional fusion, regularization, optimization, evaluation, and visualization. It does not import code from `utils.py`, `networks.py`, or any script in `synthetic/`, and it does not run or discuss competing inversion methods.

Only the three input data files are external. They contain the synthetic model, receiver geometry, and precomputed Newtonian gravity kernel.

## Problem definition and notation from the paper

Let $\mathbf{M}\in\mathbb{R}^{X\times Y\times Z}$ be the three-dimensional density-contrast model and let $\mathbf{m}=\operatorname{vec}_a(\mathbf{M})\in\mathbb{R}^{N_m}$ contain its active subsurface cells. The $N_d$ gravity observations form $\mathbf{d}\in\mathbb{R}^{N_d}$, and $\mathbf{K}\in\mathbb{R}^{N_d\times N_m}$ is the gravity sensitivity matrix. The paper's forward equation is

$$\mathbf{d}=\mathbf{K}\mathbf{m}.$$

For Cerro Machín, $N_m=89{,}413$ active cells and $N_d=676$ observations, so $N_m\gg N_d$ and the inverse problem is strongly underdetermined.

Our depth-aware implicit representation is

$$\hat{\mathbf{M}}(\boldsymbol{\Theta})=\rho_{\max}\tanh\!\left(\sum_{i=1}^{P}g_i\mathbf{W}_i\odot\mathbf{U}_i\right),$$

where $\boldsymbol{\Theta}=\{\boldsymbol{\theta}_i\}_{i=1}^{P}$ contains the parameters of $P$ independent coordinate networks. Each normalized slab field $\mathbf{U}_i$ is localized by an overlapping window $\mathbf{W}_i$ and scaled by a fixed physics-based gain $g_i$. The networks are initialized randomly and optimized for this single gravity survey; no training dataset or pretrained checkpoint is used. The true model is used only to synthesize $\mathbf d$ and evaluate the final result.

## Requirements and execution modes

Use Python 3 with `numpy`, `scipy`, `matplotlib`, and `torch`. Open the notebook from the repository or its `synthetic/` directory and run all cells in order.

### Required data placement

Keep the notebook and its three data files in the following repository structure:

```text
grav_paper/
├── data_obs_exps/
│   └── modelA/
│       ├── grav_modelA.npz
│       ├── Kernel_G.npy
│       └── receivers_modelA.npy
└── synthetic/
    └── cerro_machin_synthetic_experiment.ipynb
```

Therefore, relative to this notebook, the required data directory is `../data_obs_exps/modelA/`. Do not rename the three files. The notebook automatically locates the repository root when Jupyter is launched from either `grav_paper/` or `grav_paper/synthetic/`.

The default `paper` mode uses the full experiment: 14 slabs, width 128, four hidden layers, 64 Fourier features per slab, and 3500 optimization steps. A CUDA GPU is strongly recommended. To verify the complete workflow quickly, launch Jupyter with `CERRO_MACHIN_MODE=quick`; quick mode uses a smaller network and three steps, so its metrics are not scientific results.

Set `CERRO_MACHIN_SAVE_WEIGHTS=1` to save the trained state dictionary. All generated files go to `synthetic/notebook_results/`; inputs are never modified.

In [ ]:
# Standard-library and third-party imports — no repository Python modules are imported.
import os
import time
import platform
from pathlib import Path
from types import SimpleNamespace

os.environ.setdefault("MPLCONFIGDIR", "/tmp/grav_paper_matplotlib")

import numpy as np
import scipy
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Required placement: <repository>/data_obs_exps/modelA/<three files below>.
REQUIRED_DATA_FILES = ("grav_modelA.npz", "Kernel_G.npy", "receivers_modelA.npy")
candidate_roots = (Path.cwd(), *Path.cwd().parents)
ROOT = next(
    (p for p in candidate_roots
     if all((p / "data_obs_exps/modelA" / name).is_file() for name in REQUIRED_DATA_FILES)),
    None,
)
if ROOT is None:
    expected = Path.cwd() / "data_obs_exps/modelA"
    required_list = "\n".join(f"  - {name}" for name in REQUIRED_DATA_FILES)
    raise FileNotFoundError(
        "Cerro Machín input data were not found. Place these files in "
        f"<repository>/data_obs_exps/modelA/:\n{required_list}\n"
        f"For example, if the current directory is the repository root: {expected}"
    )

DATA_DIR = ROOT / "data_obs_exps/modelA"
OUTPUT_DIR = ROOT / "synthetic/notebook_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID = (50, 50, 50)
RHO_MAX = 400.0
SI_TO_MGAL = 1e5  # applied once when the stored kernel is loaded
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Repository : {ROOT}")
print(f"Python     : {platform.python_version()}")
print(f"NumPy      : {np.__version__}")
print(f"SciPy      : {scipy.__version__}")
print(f"PyTorch    : {torch.__version__}")
print(f"Device     : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU        : {torch.cuda.get_device_name(0)}")

## 1. Cerro Machín configuration from the paper

| Quantity | Paper value |
| --- | ---: |
| Grid $(X\times Y\times Z)$ | $50\times50\times50$ |
| Observations $N_d$ | 676 |
| Slabs $P$ | 14 |
| Window factor $\kappa$ | 0.7 |
| Fourier frequencies $L$ | 64 |
| Bandwidth range | $\sigma_{\min}=1$, $\sigma_{\max}=48$ |
| Hidden width / layers | 128 / 4 |
| Depth exponent $\beta$ | 3 |
| Depth offset $z_0$ | 3 |
| $\lambda_{\mathrm{TV}}$ / $\lambda_{L_1}$ / $\lambda_{\mathrm c}$ | 0.1 / 0.1 / 24 |
| Schedule decay rate $r$ | 10 |
| Learning rate | $10^{-3}$ |
| Iterations $T$ | 3500 |
| Random seed | 0 |

These values are stated explicitly below so the executable configuration and the paper remain synchronized.

In [ ]:
MODE = os.environ.get("CERRO_MACHIN_MODE", "paper").strip().lower()
if MODE not in {"paper", "quick"}:
    raise ValueError("CERRO_MACHIN_MODE must be 'paper' or 'quick'.")

config = SimpleNamespace(
    slabs=14, window_factor=0.7, sigma_min=1.0, sigma_max=48.0,
    epochs=3500, learning_rate=1e-3, decay_rate=10.0,
    lambda_tv=0.1, lambda_l1=0.1, consensus_max=24.0,
    z0=3.0, beta=3.0, hidden_width=128, hidden_layers=4,
    fourier_features=64, seed=0, print_every=500,
)

if MODE == "quick":
    # Smoke test only; the method and loss are unchanged.
    config.slabs = 3
    config.sigma_max = 8.0
    config.epochs = 3
    config.hidden_width = 24
    config.hidden_layers = 2
    config.fourier_features = 12
    config.consensus_max = 2.0
    config.print_every = 1

SAVE_WEIGHTS = os.environ.get("CERRO_MACHIN_SAVE_WEIGHTS", "0").lower() in {"1", "true", "yes"}
MODEL_PATH = OUTPUT_DIR / f"cerro_machin_compositional_dip_{MODE}.npy"
WEIGHTS_PATH = OUTPUT_DIR / f"cerro_machin_compositional_dip_{MODE}.pth"

print(f"Mode               : {MODE}")
print(f"Slabs              : {config.slabs}")
print(f"Optimization steps : {config.epochs}")
print(f"Hidden width/layers: {config.hidden_width}/{config.hidden_layers}")
print(f"Fourier features   : {config.fourier_features} per slab")
print(f"Output directory   : {OUTPUT_DIR}")
if MODE == "quick":
    print("WARNING: quick-mode output is only an execution test.")

## 2. Load the model and generate synthetic observations

`grav_modelA.npz` stores cell centers in metres and density contrast in g/cm³. Density is converted to kg/m³. Following the experiment definition, positive values below 200 kg/m³ and negative values above −250 kg/m³ are set to zero. NaNs mark cells above topography and define the inactive-cell mask.

The stored kernel has one column per active cell. It is converted to mGal when loaded; the resulting array is $\mathbf K$ everywhere else in the notebook. Multiplying $\mathbf K$ by the thresholded true density generates the noise-free observation vector used for self-supervised inversion.

In [ ]:
def load_cerro_machin_data(data_dir):
    model_file = np.load(data_dir / "grav_modelA.npz")
    true_density = 1000.0 * model_file["Grav_model"].astype(np.float32)
    true_density[(true_density > 0) & (true_density < 200)] = 0
    true_density[(true_density < 0) & (true_density > -250)] = 0

    inactive = np.isnan(true_density)
    kernel = np.load(data_dir / "Kernel_G.npy").astype(np.float32) * SI_TO_MGAL
    receivers = np.load(data_dir / "receivers_modelA.npy").astype(np.float32)
    observed = kernel @ true_density[~inactive]
    cell_sizes = tuple(float(model_file[name]) for name in ("dx", "dy", "dz"))
    return true_density, inactive, kernel, receivers, observed, cell_sizes

mtrue, inactive, kernel, receivers, d_obs, cell_sizes = load_cerro_machin_data(DATA_DIR)
active = ~inactive

assert mtrue.shape == (np.prod(GRID),)
assert kernel.shape == (receivers.shape[0], int(active.sum()))
assert d_obs.shape == (receivers.shape[0],)
assert np.isfinite(kernel).all() and np.isfinite(d_obs).all()

print(f"Grid            : {GRID} ({np.prod(GRID):,} cells)")
print(f"Cell dimensions : {cell_sizes} m")
print(f"Active cells    : {active.sum():,}")
print(f"Receivers       : {receivers.shape[0]}")
print(f"Kernel K        : {kernel.shape} (maps kg/m³ to mGal)")
print(f"Density range   : {np.nanmin(mtrue):.1f} to {np.nanmax(mtrue):.1f} kg/m³")
print(f"Gravity range   : {d_obs.min():.3f} to {d_obs.max():.3f} mGal")

In [ ]:
# Visualize the synthetic inputs. Array Z increases upward in this mesh.
true3d = mtrue.reshape(GRID)
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
sections = [
    (true3d[15, :, :], "XY: Z index 15", "X", "Y"),
    (true3d[:, 27, :], "XZ: Y index 27", "X", "Z index (upward)"),
    (true3d[:, :, 23], "YZ: X index 23", "Y", "Z index (upward)"),
]
for ax, (section, title, xlabel, ylabel) in zip(axes[:3], sections):
    image = ax.imshow(np.ma.masked_invalid(section), origin="lower", cmap="coolwarm",
                      vmin=-RHO_MAX, vmax=RHO_MAX)
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
scatter = axes[3].scatter(receivers[:, 0] / 1000, receivers[:, 1] / 1000,
                          c=d_obs, cmap="viridis", s=35)
axes[3].set(title="Noise-free gravity", xlabel="Easting (km)", ylabel="Northing (km)")
fig.colorbar(image, ax=axes[:3], label="density contrast (kg/m³)", shrink=0.82)
fig.colorbar(scatter, ax=axes[3], label="gravity (mGal)", shrink=0.82)
fig.suptitle("Cerro Machín synthetic model and observations")
fig.savefig(OUTPUT_DIR / "01_cerro_machin_inputs.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Mathematical formulation and complete implementation

### Slab parametrization and partition of unity

Before amplitude scaling and final bounding, the model is decomposed into $P$ overlapping slab fields:

$$\mathbf M=\sum_{i=1}^{P}\mathbf W_i\odot\mathbf U_i.$$

For slab centers $c_i=(i-1)(Z-1)/(P-1)$ and width $\sigma_w=\kappa Z/P$,

$$\phi_i(z)=\exp\!\left[-\frac{(z-c_i)^2}{2\sigma_w^2}\right],\qquad \mathbf W_i[x,y,z]=\frac{\phi_i(z)}{\sum_{j=1}^{P}\phi_j(z)}.$$

Consequently, $\sum_i\mathbf W_i[x,y,z]=1$ at every voxel depth.

### Slab-specific Fourier-feature INRs

Let $\boldsymbol{\mathcal R}=\{\boldsymbol r_n\}_{n=1}^{N}$ be all normalized grid coordinates, with $\boldsymbol r=(x,y,z)\in[0,1]^3$. Slab $i$ is

$$\mathbf U_i=\left\{\tanh\!\left[f_{\boldsymbol{\theta}_i}\!\left(\gamma_{\sigma_i}(\boldsymbol r)\right)\right]\;\middle|\;\boldsymbol r\in\boldsymbol{\mathcal R}\right\},$$

$$\gamma_{\sigma_i}(\boldsymbol r)=\left[\cos(2\pi\mathbf B_i\boldsymbol r),\sin(2\pi\mathbf B_i\boldsymbol r)\right],\qquad \mathbf B_i\in\mathbb R^{L\times3},\quad (\mathbf B_i)_{jk}\sim\mathcal N(0,\sigma_i^2).$$

The bandwidths are geometrically spaced,

$$\sigma_i=\sigma_{\min}\left(\frac{\sigma_{\max}}{\sigma_{\min}}\right)^{(i-1)/(P-1)}.$$

In the Cerro Machín array ordering, low-index slabs are deeper and receive the smaller bandwidths; shallow slabs receive the larger bandwidths and can represent finer detail, exactly as described in the paper.

### Physics-based depth gain

Let $\mathcal I_z$ contain the active kernel columns belonging to layer $z$. The mean layer sensitivity and detected surface are

$$s(z)=\frac{1}{|\mathcal I_z|}\sum_{j\in\mathcal I_z}\|\mathbf K[:,j]\|_2,\qquad z_{\mathrm{surf}}=\arg\max_z s(z).$$

With $b(z)=|z-z_{\mathrm{surf}}|$, the inverse-sensitivity gain and slab gains are

$$\operatorname{gain}(z)=\left(\frac{b(z)+z_0}{z_0}\right)^{\beta/2},\qquad g_i=\frac{\sum_z w_i(z)\operatorname{gain}(z)}{\sum_z w_i(z)},\qquad g_i\leftarrow\frac{g_i}{\min_jg_j}.$$

### Fusion and scheduled objective

The physical-density reconstruction and its normalized form are

$$\hat{\mathbf M}(\boldsymbol\Theta)=\rho_{\max}\tanh\!\left(\sum_{i=1}^{P}g_i\mathbf W_i\odot\mathbf U_i\right),\qquad \widetilde{\mathbf M}=\hat{\mathbf M}/\rho_{\max}.$$

The code uses the mean-squared form of the paper's squared $\ell_2$ data term (the factor $1/N_d$ fixes its scale relative to the reported hyperparameters):

$$\boldsymbol\Theta^*=\arg\min_{\boldsymbol\Theta}\;\frac{1}{N_d}\|\mathbf K\operatorname{vec}_a(\hat{\mathbf M})-\mathbf d\|_2^2+\lambda_{\mathrm{TV}}(t)\operatorname{TV}(\widetilde{\mathbf M})+\lambda_{L_1}(t)\|\widetilde{\mathbf M}\|_1+\lambda_{\mathrm c}(t)\mathcal C(\boldsymbol\Theta).$$

Anisotropic TV is the sum of the mean absolute forward differences along the three spatial axes. Neighboring-slab disagreement is

$$\mathcal C(\boldsymbol\Theta)=\sum_{i=1}^{P-1}\frac{1}{N}\sum_{(x,y,z)\in\boldsymbol{\mathcal R}}\left(\mathbf U_i[x,y,z]-\mathbf U_{i+1}[x,y,z]\right)^2.$$

Finally, for iteration $t=0,\ldots,T-1$,

$$\lambda_{\mathrm{TV}}(t)=\frac{\lambda_{\mathrm{TV}}}{1+rt},\qquad \lambda_{L_1}(t)=\frac{\lambda_{L_1}}{1+rt},\qquad \lambda_{\mathrm c}(t)=\lambda_{\mathrm c}\frac{t}{T-1}.$$

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_coordinates(grid, device):
    nz, ny, nx = grid
    zz, yy, xx = np.meshgrid(
        np.linspace(0, 1, nz), np.linspace(0, 1, ny), np.linspace(0, 1, nx), indexing="ij"
    )
    coords = np.stack([zz.ravel(), yy.ravel(), xx.ravel()], axis=1).astype(np.float32)
    return torch.tensor(coords, device=device)

def encode_fourier(coords, basis):
    projection = 2.0 * np.pi * coords @ basis.T
    return torch.cat([torch.cos(projection), torch.sin(projection)], dim=1)

class FourierMLP(nn.Module):
    def __init__(self, input_size, hidden_width, hidden_layers):
        super().__init__()
        layers = [nn.Linear(input_size, hidden_width), nn.SiLU()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(hidden_width, hidden_width), nn.SiLU()])
        layers.append(nn.Linear(hidden_width, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, features):
        return self.network(features)

class AnisotropicTV(nn.Module):
    def forward(self, volume):
        dz = volume[:, :, 1:, :, :] - volume[:, :, :-1, :, :]
        dy = volume[:, :, :, 1:, :] - volume[:, :, :, :-1, :]
        dx = volume[:, :, :, :, 1:] - volume[:, :, :, :, :-1]
        return dz.abs().mean() + dy.abs().mean() + dx.abs().mean()

In [ ]:
def build_depth_windows_and_gains(kernel, inactive, slabs, z0, beta, overlap=0.7):
    """Return normalized Gaussian windows, fixed depth gains, and detected surface index."""
    column_norm = np.linalg.norm(kernel, axis=0).astype(np.float64)
    full_norm = np.full(np.prod(GRID), np.nan, dtype=np.float64)
    full_norm[~inactive] = column_norm
    sensitivity = full_norm.reshape(GRID)
    valid = np.isfinite(sensitivity)
    counts = valid.sum(axis=(1, 2))
    sums = np.where(valid, sensitivity, 0.0).sum(axis=(1, 2))
    layer_mean = np.divide(sums, counts, out=np.full(GRID[0], np.nan), where=counts > 0)
    surface_index = int(np.nanargmax(layer_mean))

    z = np.arange(GRID[0])
    depth_cells = np.abs(z - surface_index)
    inverse_sensitivity = (1.0 + depth_cells / z0) ** (beta / 2.0)
    centers = np.linspace(0, GRID[0] - 1, slabs)
    width = overlap * GRID[0] / slabs
    bumps = np.exp(-((z[None, :] - centers[:, None]) ** 2) / (2.0 * width**2))
    windows = bumps / (bumps.sum(axis=0, keepdims=True) + 1e-12)
    gains = (windows * inverse_sensitivity[None, :]).sum(axis=1) / windows.sum(axis=1)
    gains /= gains.min()
    return (torch.tensor(windows, dtype=torch.float32, device=DEVICE),
            torch.tensor(gains, dtype=torch.float32, device=DEVICE), surface_index)

class CompositionalDIP(nn.Module):
    def __init__(self, coords, sigmas, windows, gains, fourier_features, hidden_width, hidden_layers):
        super().__init__()
        self.slabs = len(sigmas)
        self.register_buffer("windows", windows.view(self.slabs, 1, 1, GRID[0], 1, 1))
        self.register_buffer("gains", gains)
        self.feature_names = []
        for i, sigma in enumerate(sigmas):
            basis = torch.randn(fourier_features, 3, device=DEVICE) * sigma
            features = encode_fourier(coords, basis)
            name = f"features_{i}"
            self.register_buffer(name, features)
            self.feature_names.append(name)
        self.networks = nn.ModuleList([
            FourierMLP(2 * fourier_features, hidden_width, hidden_layers)
            for _ in range(self.slabs)
        ])

    def synthesize(self):
        raw_fields = [
            torch.tanh(self.networks[i](getattr(self, self.feature_names[i]))).view(1, 1, *GRID)
            for i in range(self.slabs)
        ]
        fused = sum(self.gains[i] * self.windows[i] * raw_fields[i] for i in range(self.slabs))
        return RHO_MAX * torch.tanh(fused), raw_fields

    def forward(self):
        return self.synthesize()[0]

In [ ]:
def train_compositional_dip(kernel, observed, inactive, cfg):
    set_seed(cfg.seed)
    coordinates = make_coordinates(GRID, DEVICE)
    sigmas = tuple(np.geomspace(cfg.sigma_min, cfg.sigma_max, cfg.slabs).round(2))
    windows, gains, surface_index = build_depth_windows_and_gains(
        kernel, inactive, cfg.slabs, cfg.z0, cfg.beta, cfg.window_factor
    )
    model = CompositionalDIP(
        coordinates, sigmas, windows, gains, cfg.fourier_features,
        cfg.hidden_width, cfg.hidden_layers
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.01, total_iters=cfg.epochs
    )
    tv = AnisotropicTV().to(DEVICE)
    kernel_t = torch.tensor(kernel, dtype=torch.float32, device=DEVICE)
    observed_t = torch.tensor(observed, dtype=torch.float32, device=DEVICE)
    inactive_t = torch.tensor(inactive, dtype=torch.bool, device=DEVICE)

    history = {name: [] for name in ("total", "data", "tv", "l1", "consensus")}
    best_loss, best_volume, best_state = np.inf, None, None
    for step in range(cfg.epochs):
        optimizer.zero_grad()
        volume, raw_fields = model.synthesize()
        active_density = volume.reshape(-1)[~inactive_t]
        predicted = kernel_t @ active_density
        loss_data = torch.mean((predicted - observed_t) ** 2)

        decay = 1.0 / (1.0 + cfg.decay_rate * step)
        loss_tv = cfg.lambda_tv * decay * tv(volume / RHO_MAX)
        loss_l1 = cfg.lambda_l1 * decay * torch.norm(volume / RHO_MAX, p=1)
        consensus_weight = cfg.consensus_max * step / max(cfg.epochs - 1, 1)
        loss_consensus = consensus_weight * sum(
            torch.mean((raw_fields[i] - raw_fields[i + 1]) ** 2)
            for i in range(len(raw_fields) - 1)
        )
        total_loss = loss_data + loss_tv + loss_l1 + loss_consensus
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        values = {"total": total_loss.item(), "data": loss_data.item(),
                  "tv": loss_tv.item(), "l1": loss_l1.item(),
                  "consensus": loss_consensus.item()}
        for name, value in values.items():
            history[name].append(value)
        if values["total"] < best_loss:
            best_loss = values["total"]
            best_volume = volume.detach().cpu().numpy().copy()
            best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        if step % cfg.print_every == 0 or step == cfg.epochs - 1:
            regularization = values["tv"] + values["l1"] + values["consensus"]
            print(f"step {step:04d}/{cfg.epochs - 1:04d} | data {values['data']:.4e} | regularization {regularization:.4e}")

    model.load_state_dict(best_state)
    diagnostics = {"history": history, "sigmas": np.asarray(sigmas),
                   "windows": windows.detach().cpu().numpy(),
                   "gains": gains.detach().cpu().numpy(),
                   "surface_index": surface_index, "best_loss": best_loss}
    return best_volume, model, diagnostics

## 4. Inspect the fixed depth prior

For this mesh, Z index increases from deep cells toward topography. The sensitivity-defined surface and gain construction do not assume that orientation: the kernel determines the surface layer. The window profiles must sum to one at every Z layer.

In [ ]:
preview_windows, preview_gains, preview_surface = build_depth_windows_and_gains(
    kernel, inactive, config.slabs, config.z0, config.beta, config.window_factor
)
preview_windows = preview_windows.cpu().numpy()
preview_gains = preview_gains.cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, profile in enumerate(preview_windows):
    axes[0].plot(np.arange(GRID[0]), profile, label=f"slab {i + 1}")
axes[0].axvline(preview_surface, color="black", linestyle="--", label="detected surface")
axes[0].set(xlabel="Z layer index (deep → top)", ylabel=r"window $W_i(z)$",
            title="Overlapping depth windows")
axes[0].grid(alpha=0.2)
axes[1].plot(np.arange(1, config.slabs + 1), preview_gains, "o-")
axes[1].set(xlabel="slab in grid order (deep → top)", ylabel=r"gain $g_i$",
            title="Fixed inverse-sensitivity gains")
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"02_depth_prior_{MODE}.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Detected surface Z index     : {preview_surface}")
print(f"Partition-of-unity max error: {np.abs(preview_windows.sum(0) - 1).max():.3e}")
print("Gains (deep → top):", np.array2string(preview_gains, precision=2))

## 5. Train our reconstruction

This is the expensive cell in paper mode. It performs a complete forward and backward pass through all slab networks at each step and selects the state with the smallest total objective. The saved NumPy array is flat in standard C order and can be restored with `reshape((50, 50, 50))`.

In [ ]:
print(f"Training our compositional DIP on {DEVICE} ...")
start = time.perf_counter()
reconstruction, trained_model, diagnostics = train_compositional_dip(kernel, d_obs, inactive, config)
elapsed = time.perf_counter() - start
np.save(MODEL_PATH, reconstruction.squeeze().ravel().astype(np.float32))
if SAVE_WEIGHTS:
    torch.save(trained_model.state_dict(), WEIGHTS_PATH)
print(f"Training time       : {elapsed:.1f} s ({elapsed / 60:.1f} min)")
print(f"Best total objective: {diagnostics['best_loss']:.6e}")
print(f"Saved reconstruction: {MODEL_PATH}")
if SAVE_WEIGHTS:
    print(f"Saved weights       : {WEIGHTS_PATH}")
assert reconstruction.shape == (1, 1) + GRID
assert np.isfinite(reconstruction).all()

In [ ]:
history = diagnostics["history"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, label in [("total", "total"), ("data", "data MSE"),
                    ("tv", "TV"), ("l1", "L1"), ("consensus", "consensus")]:
    ax.semilogy(np.maximum(history[name], 1e-16), label=label)
ax.set(xlabel="optimization step", ylabel="loss term",
       title=f"Compositional DIP training ({MODE} mode)")
ax.grid(alpha=0.2)
ax.legend(ncol=3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"03_training_history_{MODE}.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Paper evaluation metrics

The paper evaluates the known synthetic density in the model domain using RMSE, PSNR, and three-dimensional SSIM, and evaluates the predicted measurements separately using receiver-domain RMSE. With active-cell truth $\mathbf m$ and prediction $\hat{\mathbf m}$,

$$\operatorname{RMSE}_{\mathrm{model}}=\sqrt{\frac{1}{N_m}\|\hat{\mathbf m}-\mathbf m\|_2^2},\qquad \operatorname{PSNR}=20\log_{10}\left(\frac{\rho_{\max}-\rho_{\min}}{\operatorname{RMSE}_{\mathrm{model}}}\right),$$

$$\operatorname{RMSE}_{\mathrm{receiver}}=\sqrt{\frac{1}{N_d}\|\mathbf K\hat{\mathbf m}-\mathbf d\|_2^2}.$$

Here $\rho_{\min}=-400$ and $\rho_{\max}=400$ kg/m³, giving a PSNR range of 800 kg/m³. SSIM uses a 7×7×7 uniform window and the standard constants $C_1=(0.01L)^2$ and $C_2=(0.03L)^2$, with $L=800$ kg/m³. The true density is accessed only in this post-training evaluation section.

In [ ]:
def ssim_3d(prediction, truth, inactive, data_range=800.0, window_size=7):
    x = np.asarray(prediction, dtype=np.float64).reshape(GRID).copy()
    y = np.asarray(truth, dtype=np.float64).reshape(GRID).copy()
    mask = inactive.reshape(GRID)
    x[mask] = 0.0
    y[mask] = 0.0
    ux, uy = uniform_filter(x, window_size), uniform_filter(y, window_size)
    uxx, uyy = uniform_filter(x * x, window_size), uniform_filter(y * y, window_size)
    uxy = uniform_filter(x * y, window_size)
    correction = window_size**3 / (window_size**3 - 1)
    vx, vy = correction * (uxx - ux * ux), correction * (uyy - uy * uy)
    vxy = correction * (uxy - ux * uy)
    c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
    score = ((2 * ux * uy + c1) * (2 * vxy + c2)) / (
        (ux * ux + uy * uy + c1) * (vx + vy + c2))
    pad = (window_size - 1) // 2
    return float(score[pad:-pad, pad:-pad, pad:-pad].mean())

def evaluate_reconstruction(prediction):
    pred_active = np.asarray(prediction).reshape(-1)[active].astype(np.float64)
    true_active = mtrue[active].astype(np.float64)
    mse = np.mean((pred_active - true_active) ** 2)
    predicted_data = kernel @ pred_active
    metrics = {
        "RMSE (kg/m³)": float(np.sqrt(mse)),
        "PSNR (dB)": float(20 * np.log10(2 * RHO_MAX) - 10 * np.log10(mse + 1e-12)),
        "3D SSIM": ssim_3d(prediction, mtrue, inactive),
        "receiver RMSE (mGal)": float(np.sqrt(np.mean((predicted_data - d_obs) ** 2))),
    }
    return metrics, predicted_data

metrics, d_pred = evaluate_reconstruction(reconstruction)
print("Our compositional DIP — Cerro Machín synthetic experiment")
for name, value in metrics.items():
    print(f"{name:28s}: {value:.6g}")
if MODE == "quick":
    print("\nQuick-mode metrics are not scientific results; run paper mode for reproduction.")

In [ ]:
# True and reconstructed sections on one common density scale.
recon3d = reconstruction.reshape(GRID).copy()
active3d = active.reshape(GRID)
recon3d[~active3d] = np.nan
volumes = [("true synthetic model", true3d), ("our compositional DIP", recon3d)]
slicers = [
    (lambda v: v[15, :, :], "XY, Z=15", "X", "Y"),
    (lambda v: v[:, 27, :], "XZ, Y=27", "X", "Z index (upward)"),
    (lambda v: v[:, :, 23], "YZ, X=23", "Y", "Z index (upward)"),
]
fig, axes = plt.subplots(2, 3, figsize=(12, 7.5), constrained_layout=True)
for row, (name, volume) in enumerate(volumes):
    for col, (take, title, xlabel, ylabel) in enumerate(slicers):
        image = axes[row, col].imshow(np.ma.masked_invalid(take(volume)), origin="lower",
                                     cmap="coolwarm", vmin=-RHO_MAX, vmax=RHO_MAX)
        if row == 0:
            axes[row, col].set_title(title)
        axes[row, col].set_xlabel(xlabel)
        axes[row, col].set_ylabel(f"{name}\n{ylabel}" if col == 0 else ylabel)
fig.colorbar(image, ax=axes, shrink=0.78, label="density contrast (kg/m³)")
fig.suptitle(f"Cerro Machín synthetic reconstruction ({MODE} mode)")
fig.savefig(OUTPUT_DIR / f"04_reconstruction_sections_{MODE}.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Depth distribution and predicted-versus-observed gravity.
def normalized_depth_profile(volume):
    values = np.abs(np.asarray(volume).reshape(GRID).copy())
    values[~active3d] = 0.0
    profile = values.sum(axis=(1, 2))
    return profile / (profile.sum() + 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(normalized_depth_profile(mtrue), label="true synthetic model")
axes[0].plot(normalized_depth_profile(reconstruction), label="our compositional DIP")
axes[0].axvline(diagnostics["surface_index"], color="black", linestyle="--", label="surface")
axes[0].set(xlabel="Z layer index (deep → top)", ylabel=r"normalized $|m|$ per layer",
            title="Recovered depth distribution")
axes[0].grid(alpha=0.2)
axes[0].legend()
limits = [min(d_obs.min(), d_pred.min()), max(d_obs.max(), d_pred.max())]
axes[1].plot(limits, limits, "k--", label="perfect prediction")
axes[1].scatter(d_obs, d_pred, s=16, alpha=0.55, label="our prediction")
axes[1].set(xlabel="observed gravity (mGal)", ylabel="predicted gravity (mGal)",
            title="Gravity-data agreement")
axes[1].grid(alpha=0.2)
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"05_depth_and_data_fit_{MODE}.png", dpi=160, bbox_inches="tight")
plt.show()

## Interpretation

A successful paper-mode run should reduce the gravity-data loss while producing coherent signed bodies and a vertical density distribution close to the synthetic truth. The paper reports **27.15 dB PSNR, 35.12 kg/m³ model RMSE, 0.873 SSIM, and 0.071 mGal receiver RMSE** for the Cerro Machín reconstruction. Small numerical differences can occur across software and hardware environments. Receiver fit must be interpreted together with the three model-domain metrics and the section plots because gravity inversion is underdetermined.

Reproducibility checks:

- confirm the kernel shape is `(676, 89413)`;
- confirm seed 0 and the printed paper configuration;
- confirm the window partition error is close to machine precision;
- use a CUDA GPU with sufficient memory for the full run;
- expect small floating-point differences across PyTorch, CUDA, and hardware versions.

Quick mode only confirms that every part of our method executes. It is not expected to reconstruct the volcanic bodies in three optimization steps.